In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from recommenders.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float64

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [3]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [4]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
# used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100

In [5]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
R = matrix_mf.R
R

tensor([[  0.,   0.,   0.,  ...,   0.,   0.,   0.],
        [  0.,   0.,   0.,  ...,   0.,   0.,   0.],
        [  0.,   0.,   0.,  ...,   0.,   0.,   0.],
        ...,
        [ 80.,   0.,   0.,  ...,   0.,   0.,  80.],
        [ 50., 180.,  50.,  ...,   0.,  50.,   0.],
        [  0.,   0.,   0.,  ...,   0.,   0.,   0.]], device='mps:0')

In [6]:
alpha = matrix_mf.compute_alpha().item()
R *= alpha
alpha

0.09144702553749084

In [7]:
num_latent_factors = 100
lmf = LogisticMatrixFactorization(
    R=R,
    num_factors=num_latent_factors,
    alpha=alpha,
    lambd=0.01,
    device=device,
    dtype=dtype,
)

num_epochs = 10000
lmf.train_with_gradients(
    num_epochs=num_epochs,
    learning_rate=0.01,
    log_interval=10,
)

Epoch 1: loss = 107504.6484375, MPR = 0.49529629945755005
Epoch 11: loss = 68812.0, MPR = 0.4382598102092743
Epoch 21: loss = 65343.12890625, MPR = 0.4032478928565979
Epoch 31: loss = 54251.6796875, MPR = 0.37432730197906494
Epoch 41: loss = 43614.390625, MPR = 0.34907814860343933
Epoch 51: loss = 51348.34375, MPR = 0.32703930139541626
Epoch 61: loss = 70398.1953125, MPR = 0.3070744574069977
Epoch 71: loss = 39351.3515625, MPR = 0.2885867655277252
Epoch 81: loss = 41856.76953125, MPR = 0.272062212228775
Epoch 91: loss = 33016.640625, MPR = 0.25723353028297424
Epoch 101: loss = 35650.76953125, MPR = 0.24339456856250763
Epoch 111: loss = 36313.765625, MPR = 0.23067109286785126
Epoch 121: loss = 58521.7890625, MPR = 0.21964555978775024
Epoch 131: loss = 23597.244140625, MPR = 0.2088986188173294
Epoch 141: loss = 30619.607421875, MPR = 0.1989588439464569
Epoch 151: loss = 54841.83984375, MPR = 0.1902640461921692
Epoch 161: loss = 29517.71875, MPR = 0.1821192502975464
Epoch 171: loss = 2524

In [8]:
lmf.save("models", "lmf_all_types_100")
lmf = LogisticMatrixFactorization.load(os.path.join("models", "lmf_all_types_100.pt"))

In [9]:
px.line(x=range(len(lmf.losses)), y=lmf.losses.cpu(), title="Loss").show()
px.line(x=range(len(lmf.mprs)), y=lmf.mprs.cpu(), title="MPRS").show()

In [13]:
user_id = matrix_mf.usernames_to_ids(["michelle"])[0]

# Get the top 10 recommendations for the user
top_10_ids, top_10_ids_scores = lmf.recommend(user_id, top_k=20, filter_user_items=True)

# Get the top 10 recommendations for the user
matrix_mf.items_ids_to_df(top_10_ids)[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
112,jaslkh,Fleetwood Mac,Dreams - 2004 Remaster,1977,89,0.828,0.492,0.0276,0.0644,0.004280,0.128,0.789,120.151,-9.744,257800,1977,89
282,jaslkh,Iliona,Si tu m'aimes demain,2022,59,0.854,0.501,0.0310,0.6320,0.001190,0.103,0.465,105.011,-9.535,176026,2022,59
359,jaslkh,Jacques Brel,Ces gens-là,1988,49,0.531,0.210,0.1640,0.6260,0.000000,0.163,0.494,129.663,-18.451,278400,1988,49
409,jaslkh,Fleetwood Mac,Dreams - 2004 Remaster,1977,89,0.828,0.492,0.0276,0.0644,0.004280,0.128,0.789,120.151,-9.744,257800,1977,89
914,jaslkh,Iliona,Si tu m'aimes demain,2022,59,0.854,0.501,0.0310,0.6320,0.001190,0.103,0.465,105.011,-9.535,176026,2022,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10531,paul,Josman,L'Occasion,2018,47,0.799,0.593,0.3730,0.2690,0.000000,0.117,0.714,85.995,-5.865,223255,2018,47
10532,paul,Josman,L'Occasion,2018,47,0.799,0.593,0.3730,0.2690,0.000000,0.117,0.714,85.995,-5.865,223255,2018,47
10756,paul,Téléphone,Cendrillon,2004,46,0.507,0.489,0.0326,0.3010,0.019200,0.105,0.288,147.932,-13.273,238333,2004,46
10864,paul,PNL,Le monde ou rien,2015,64,0.648,0.465,0.1710,0.7720,0.000002,0.287,0.407,103.025,-9.716,256280,2015,64


In [11]:
if lmf.num_factors <= 3:
    df_tracks = df_matrix_mf.copy()
    df_tracks = df_tracks.sample(frac=1) # Shuffle the dataframe
    df_tracks = df_tracks[:1000] # Keep only 1000 tracks

    # Add item latent factors to the dataframe
    items_latent_columns = [f"track_latent_{i}" for i in range(lmf.num_factors)]
    for track_id in df_tracks["id"].unique():
        latent_factors = lmf.get_item_latent_factors(matrix_mf.items_to_ids([track_id])[0]).tolist()
        df_tracks.loc[df_tracks["id"] == track_id, items_latent_columns] = latent_factors

    # Add user latent factors to the dataframe
    users_latent_columns = [f"latent_factor_{i}" for i in range(lmf.num_factors)]
    for user_id in df_tracks["username"].unique():
        latent_factors = lmf.get_user_latent_factors(matrix_mf.usernames_to_ids([user_id])[0]).tolist()
        df_tracks.loc[df_tracks["username"] == user_id, users_latent_columns] = latent_factors

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=items_latent_columns,
    ).show()

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=users_latent_columns,
    ).show()